# 01 — Dataset Validation and Cleaning


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Voice2Venture_Final_Dataset.zip to Voice2Venture_Final_Dataset.zip


Upload and Extract Dataset

In [3]:
import zipfile
import os

zip_file = "Voice2Venture_Final_Dataset.zip"
extract_dir = "/content/voice2venture_data"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [4]:
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 4 * (level + 1)

    for file in files:
        print(f"{subindent}{file}")

voice2venture_data/
    Voice2Venture_Final_Dataset/
        test_bandit_interactions.csv
        bandit_interactions.csv
        test_business_context.csv
        README.txt
        recommendation_actions.csv
        dataset_manifest.json
        context_action_mapping.csv
        master_business_context.csv
        train_business_context.csv
        source_world_bank_aggregate_data.xlsx
        train_bandit_interactions.csv


In [5]:
from pathlib import Path

data_path = Path(extract_dir)

csv_files = list(data_path.rglob("*.csv"))
xlsx_files = list(data_path.rglob("*.xlsx"))

print("CSV files:")
for f in csv_files:
    print(f)

print("\nExcel files:")
for f in xlsx_files:
    print(f)

CSV files:
/content/voice2venture_data/Voice2Venture_Final_Dataset/test_bandit_interactions.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/bandit_interactions.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/test_business_context.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/recommendation_actions.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/context_action_mapping.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/master_business_context.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/train_business_context.csv
/content/voice2venture_data/Voice2Venture_Final_Dataset/train_bandit_interactions.csv

Excel files:
/content/voice2venture_data/Voice2Venture_Final_Dataset/source_world_bank_aggregate_data.xlsx


 Load Business Context Datasets


In [6]:
import pandas as pd

master_file = next(
    f for f in csv_files
    if f.name == "master_business_context.csv"
)

train_file = next(
    f for f in csv_files
    if f.name == "train_business_context.csv"
)

test_file = next(
    f for f in csv_files
    if f.name == "test_business_context.csv"
)

master_df = pd.read_csv(master_file)
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("Master shape:", master_df.shape)
print("Train shape :", train_df.shape)
print("Test shape  :", test_df.shape)

Master shape: (998, 31)
Train shape : (798, 31)
Test shape  : (200, 31)


Dataset Overview

In [7]:
overview = pd.DataFrame({
    "Dataset": [
        "Master",
        "Train",
        "Test"
    ],
    "Rows": [
        len(master_df),
        len(train_df),
        len(test_df)
    ],
    "Columns": [
        len(master_df.columns),
        len(train_df.columns),
        len(test_df.columns)
    ],
    "Missing Cells": [
        master_df.isnull().sum().sum(),
        train_df.isnull().sum().sum(),
        test_df.isnull().sum().sum()
    ],
    "Duplicate Rows": [
        master_df.duplicated().sum(),
        train_df.duplicated().sum(),
        test_df.duplicated().sum()
    ]
})

overview

,Dataset,Rows,Columns,Missing Cells,Duplicate Rows
0,Master,998,31,0,0
1,Train,798,31,0,0
2,Test,200,31,0,0


In [8]:
train_df.columns.tolist()

['business_id',
 'city',
 'country',
 'sector',
 'female_majority_owned',
 'migrant_owner',
 'previously_unemployed_owner',
 'primary_income_earner',
 'owner_has_contract_job',
 'household_size_proxy',
 'employee_count',
 'bank_account',
 'has_loan',
 'friends_family_finance',
 'sells_on_credit',
 'buys_on_credit',
 'mobile_money',
 'computer_or_tablet',
 'electricity',
 'digital_adoption_score',
 'profit_last_month',
 'household_based_business',
 'non_fixed_premises',
 'pl_statement',
 'management_score_proxy',
 'monthly_revenue_proxy_inr',
 'monthly_profit_proxy_inr',
 'finance_constraint',
 'customer_reach_constraint',
 'management_constraint',
 'data_origin']

In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   business_id                  798 non-null    object 
 1   city                         798 non-null    object 
 2   country                      798 non-null    object 
 3   sector                       798 non-null    object 
 4   female_majority_owned        798 non-null    int64  
 5   migrant_owner                798 non-null    int64  
 6   previously_unemployed_owner  798 non-null    int64  
 7   primary_income_earner        798 non-null    int64  
 8   owner_has_contract_job       798 non-null    int64  
 9   household_size_proxy         798 non-null    float64
 10  employee_count               798 non-null    int64  
 11  bank_account                 798 non-null    int64  
 12  has_loan                     798 non-null    int64  
 13  friends_family_finan

In [10]:
train_df.head()

,business_id,city,country,sector,female_majority_owned,migrant_owner,previously_unemployed_owner,primary_income_earner,owner_has_contract_job,household_size_proxy,...,household_based_business,non_fixed_premises,pl_statement,management_score_proxy,monthly_revenue_proxy_inr,monthly_profit_proxy_inr,finance_constraint,customer_reach_constraint,management_constraint,data_origin
0,V2V0001,Hyderabad,India,Other Services,0,1,0,1,0,4.1,...,0,0,1,41.8,52962.15,12198.63,1,1,0,Synthetic profile calibrated to World Bank 202...
1,V2V0003,Hyderabad,India,Retail,0,0,0,1,0,3.0,...,0,0,1,61.8,72080.45,36349.05,0,0,0,Synthetic profile calibrated to World Bank 202...
2,V2V0004,Hyderabad,India,Manufacturing,0,0,0,1,0,3.1,...,0,0,0,61.4,40011.08,17449.21,0,0,0,Synthetic profile calibrated to World Bank 202...
3,V2V0005,Hyderabad,India,Manufacturing,0,0,0,1,0,2.8,...,0,0,1,69.7,63358.00,3160.43,1,0,0,Synthetic profile calibrated to World Bank 202...
4,V2V0006,Hyderabad,India,Manufacturing,0,1,0,0,0,4.1,...,0,0,0,73.0,86450.06,11027.78,1,1,0,Synthetic profile calibrated to World Bank 202...


In [11]:
train_df.sample(5)

,business_id,city,country,sector,female_majority_owned,migrant_owner,previously_unemployed_owner,primary_income_earner,owner_has_contract_job,household_size_proxy,...,household_based_business,non_fixed_premises,pl_statement,management_score_proxy,monthly_revenue_proxy_inr,monthly_profit_proxy_inr,finance_constraint,customer_reach_constraint,management_constraint,data_origin
657,V2V0817,Surat,India,Manufacturing,0,0,0,1,0,3.1,...,0,0,0,50.8,119937.17,20765.59,1,1,0,Synthetic profile calibrated to World Bank 202...
621,V2V0772,Surat,India,Retail,1,0,0,0,0,4.9,...,0,0,0,51.6,123021.44,23227.65,0,1,0,Synthetic profile calibrated to World Bank 202...
35,V2V0042,Hyderabad,India,Manufacturing,0,1,0,1,0,3.7,...,0,0,1,57.0,38165.46,338.83,1,1,0,Synthetic profile calibrated to World Bank 202...
435,V2V0551,Mumbai,India,Retail,0,1,0,0,0,2.8,...,0,0,0,14.1,47309.56,9549.23,1,1,1,Synthetic profile calibrated to World Bank 202...
547,V2V0681,Sehore,India,Retail,1,0,0,0,0,3.8,...,0,1,1,83.6,25128.23,8979.36,1,1,0,Synthetic profile calibrated to World Bank 202...


In [12]:
train_df.sample(5)

,business_id,city,country,sector,female_majority_owned,migrant_owner,previously_unemployed_owner,primary_income_earner,owner_has_contract_job,household_size_proxy,...,household_based_business,non_fixed_premises,pl_statement,management_score_proxy,monthly_revenue_proxy_inr,monthly_profit_proxy_inr,finance_constraint,customer_reach_constraint,management_constraint,data_origin
655,V2V0815,Surat,India,Manufacturing,0,0,1,0,0,3.3,...,0,0,0,33.4,92441.31,1665.19,1,1,1,Synthetic profile calibrated to World Bank 202...
96,V2V0126,Jaipur,India,Retail,0,0,0,1,0,4.9,...,0,0,0,45.9,192680.09,83890.11,1,1,0,Synthetic profile calibrated to World Bank 202...
593,V2V0737,Sehore,India,Manufacturing,1,0,0,1,0,5.9,...,0,1,0,100.0,39247.34,5367.87,1,1,0,Synthetic profile calibrated to World Bank 202...
211,V2V0270,Kochi,India,Other Services,0,0,1,1,1,3.4,...,0,0,0,53.4,64891.31,14278.16,1,1,0,Synthetic profile calibrated to World Bank 202...
449,V2V0568,Mumbai,India,Other Services,0,0,0,1,0,4.2,...,0,0,1,25.3,51465.11,3411.00,0,1,1,Synthetic profile calibrated to World Bank 202...


 Data Dictionary

In [13]:
data_dictionary = pd.DataFrame({
    "column": train_df.columns,
    "data_type": train_df.dtypes.astype(str).values,
    "missing_values": train_df.isnull().sum().values,
    "missing_percentage": (
        train_df.isnull().mean() * 100
    ).round(2).values,
    "unique_values": [
        train_df[col].nunique()
        for col in train_df.columns
    ]
})

data_dictionary

,column,data_type,missing_values,missing_percentage,unique_values
0,business_id,object,0,0.0,798
1,city,object,0,0.0,8
2,country,object,0,0.0,1
3,sector,object,0,0.0,3
4,female_majority_owned,int64,0,0.0,2
5,migrant_owner,int64,0,0.0,2
6,previously_unemployed_owner,int64,0,0.0,2
7,primary_income_earner,int64,0,0.0,2
8,owner_has_contract_job,int64,0,0.0,2
9,household_size_proxy,float64,0,0.0,59


In [15]:
from google.colab import files as colab_files

data_dictionary.to_csv(
    "/content/data_dictionary.csv",
    index=False
)

colab_files.download("/content/data_dictionary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Missing Value Analysis

In [18]:
missing_summary = pd.DataFrame({
    "Train Missing": train_df.isnull().sum(),
    "Train Missing %": (
        train_df.isnull().mean() * 100
    ).round(2),
    "Test Missing": test_df.isnull().sum(),
    "Test Missing %": (
        test_df.isnull().mean() * 100
    ).round(2)
})

missing_summary

,Train Missing,Train Missing %,Test Missing,Test Missing %
business_id,0,0.0,0,0.0
city,0,0.0,0,0.0
country,0,0.0,0,0.0
sector,0,0.0,0,0.0
female_majority_owned,0,0.0,0,0.0
migrant_owner,0,0.0,0,0.0
previously_unemployed_owner,0,0.0,0,0.0
primary_income_earner,0,0.0,0,0.0
owner_has_contract_job,0,0.0,0,0.0
household_size_proxy,0,0.0,0,0.0


In [19]:
missing_summary[
    (missing_summary["Train Missing"] > 0) |
    (missing_summary["Test Missing"] > 0)
]

,Train Missing,Train Missing %,Test Missing,Test Missing %


Duplicate Analysis

In [20]:
print(
    "Train duplicate rows:",
    train_df.duplicated().sum()
)

print(
    "Test duplicate rows:",
    test_df.duplicated().sum()
)

Train duplicate rows: 0
Test duplicate rows: 0


Business ID Validation

In [21]:
print(
    "Train duplicate business IDs:",
    train_df["business_id"].duplicated().sum()
)

print(
    "Test duplicate business IDs:",
    test_df["business_id"].duplicated().sum()
)

Train duplicate business IDs: 0
Test duplicate business IDs: 0


In [22]:
train_ids = set(train_df["business_id"])
test_ids = set(test_df["business_id"])

overlap = train_ids.intersection(test_ids)

print("Number of overlapping business IDs:", len(overlap))

Number of overlapping business IDs: 0


In [23]:
master_ids = set(master_df["business_id"])

print(
    "Train IDs not present in master:",
    len(train_ids - master_ids)
)

print(
    "Test IDs not present in master:",
    len(test_ids - master_ids)
)

Train IDs not present in master: 0
Test IDs not present in master: 0


Train/Test Consistency

In [24]:
combined_ids = train_ids.union(test_ids)

print(
    "Master IDs not in train/test:",
    len(master_ids - combined_ids)
)

Master IDs not in train/test: 0


In [25]:
dtype_check = pd.DataFrame({
    "Train dtype": train_df.dtypes.astype(str),
    "Test dtype": test_df.dtypes.astype(str)
})

dtype_check

,Train dtype,Test dtype
business_id,object,object
city,object,object
country,object,object
sector,object,object
female_majority_owned,int64,int64
migrant_owner,int64,int64
previously_unemployed_owner,int64,int64
primary_income_earner,int64,int64
owner_has_contract_job,int64,int64
household_size_proxy,float64,float64


Data Type Validation


In [26]:
dtype_check[
    dtype_check["Train dtype"] !=
    dtype_check["Test dtype"]
]

,Train dtype,Test dtype


In [27]:
numeric_columns = train_df.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("NUMERICAL:")
print(numeric_columns)

print("\nCATEGORICAL:")
print(categorical_columns)

NUMERICAL:
['female_majority_owned', 'migrant_owner', 'previously_unemployed_owner', 'primary_income_earner', 'owner_has_contract_job', 'household_size_proxy', 'employee_count', 'bank_account', 'has_loan', 'friends_family_finance', 'sells_on_credit', 'buys_on_credit', 'mobile_money', 'computer_or_tablet', 'electricity', 'digital_adoption_score', 'profit_last_month', 'household_based_business', 'non_fixed_premises', 'pl_statement', 'management_score_proxy', 'monthly_revenue_proxy_inr', 'monthly_profit_proxy_inr', 'finance_constraint', 'customer_reach_constraint', 'management_constraint']

CATEGORICAL:
['business_id', 'city', 'country', 'sector', 'data_origin']


Binary Feature Validation

In [28]:
binary_columns = [
    "female_majority_owned",
    "migrant_owner",
    "previously_unemployed_owner",
    "primary_income_earner",
    "owner_has_contract_job",
    "bank_account",
    "has_loan",
    "friends_family_finance",
    "sells_on_credit",
    "buys_on_credit",
    "mobile_money",
    "computer_or_tablet",
    "electricity",
    "profit_last_month",
    "household_based_business",
    "non_fixed_premises",
    "pl_statement",
    "finance_constraint",
    "customer_reach_constraint",
    "management_constraint"
]

In [29]:
for col in binary_columns:

    if col in train_df.columns:

        print(f"\n{col}")
        print(
            "Train:",
            sorted(train_df[col].dropna().unique())
        )

        print(
            "Test :",
            sorted(test_df[col].dropna().unique())
        )


female_majority_owned
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

migrant_owner
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

previously_unemployed_owner
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

primary_income_earner
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

owner_has_contract_job
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

bank_account
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

has_loan
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

friends_family_finance
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

sells_on_credit
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

buys_on_credit
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

mobile_money
Train: [np.int64(0), np.int64(1)]
Test : [np.int64(0), np.int64(1)]

computer_or_tablet
Train: [np.int64(0), np

In [30]:
binary_results = []

for col in binary_columns:

    if col in train_df.columns:

        train_invalid = (
            ~train_df[col].isin([0, 1])
            & train_df[col].notna()
        ).sum()

        test_invalid = (
            ~test_df[col].isin([0, 1])
            & test_df[col].notna()
        ).sum()

        binary_results.append({
            "column": col,
            "train_invalid": train_invalid,
            "test_invalid": test_invalid
        })

binary_validation = pd.DataFrame(binary_results)

binary_validation

,column,train_invalid,test_invalid
0,female_majority_owned,0,0
1,migrant_owner,0,0
2,previously_unemployed_owner,0,0
3,primary_income_earner,0,0
4,owner_has_contract_job,0,0
5,bank_account,0,0
6,has_loan,0,0
7,friends_family_finance,0,0
8,sells_on_credit,0,0
9,buys_on_credit,0,0


Numerical Range Validation

In [31]:
range_summary = pd.DataFrame({
    "train_min": train_df[numeric_columns].min(),
    "train_max": train_df[numeric_columns].max(),
    "test_min": test_df[numeric_columns].min(),
    "test_max": test_df[numeric_columns].max()
})

range_summary

,train_min,train_max,test_min,test_max
female_majority_owned,0.00,1.00,0.00,1.00
migrant_owner,0.00,1.00,0.00,1.00
previously_unemployed_owner,0.00,1.00,0.00,1.00
primary_income_earner,0.00,1.00,0.00,1.00
owner_has_contract_job,0.00,1.00,0.00,1.00
household_size_proxy,1.30,7.80,1.50,6.70
employee_count,1.00,5.00,1.00,4.00
bank_account,0.00,1.00,0.00,1.00
has_loan,0.00,1.00,0.00,1.00
friends_family_finance,0.00,1.00,0.00,1.00


In [32]:
negative_results = []

for col in numeric_columns:

    train_negative = (
        train_df[col] < 0
    ).sum()

    test_negative = (
        test_df[col] < 0
    ).sum()

    negative_results.append({
        "column": col,
        "train_negative": train_negative,
        "test_negative": test_negative
    })

negative_validation = pd.DataFrame(
    negative_results
)

negative_validation[
    (negative_validation["train_negative"] > 0) |
    (negative_validation["test_negative"] > 0)
]

,column,train_negative,test_negative


Financial Consistency Checks

In [33]:
financial_check = pd.DataFrame({
    "revenue_min": [
        train_df["monthly_revenue_proxy_inr"].min(),
        test_df["monthly_revenue_proxy_inr"].min()
    ],
    "profit_min": [
        train_df["monthly_profit_proxy_inr"].min(),
        test_df["monthly_profit_proxy_inr"].min()
    ],
    "revenue_max": [
        train_df["monthly_revenue_proxy_inr"].max(),
        test_df["monthly_revenue_proxy_inr"].max()
    ],
    "profit_max": [
        train_df["monthly_profit_proxy_inr"].max(),
        test_df["monthly_profit_proxy_inr"].max()
    ]
}, index=["Train", "Test"])

financial_check

,revenue_min,profit_min,revenue_max,profit_max
Train,8000.00,17.38,439707.07,214421.12
Test,11446.47,40.59,220363.31,90193.48


In [34]:
profit_greater_than_revenue_train = (
    train_df["monthly_profit_proxy_inr"] >
    train_df["monthly_revenue_proxy_inr"]
).sum()

profit_greater_than_revenue_test = (
    test_df["monthly_profit_proxy_inr"] >
    test_df["monthly_revenue_proxy_inr"]
).sum()

print(
    "Train profit > revenue:",
    profit_greater_than_revenue_train
)

print(
    "Test profit > revenue:",
    profit_greater_than_revenue_test
)

Train profit > revenue: 0
Test profit > revenue: 0


Categorical Validation


In [35]:
for col in categorical_columns:

    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)

    print("TRAIN:")
    print(train_df[col].value_counts(dropna=False))

    print("\nTEST:")
    print(test_df[col].value_counts(dropna=False))


business_id
TRAIN:
business_id
V2V0998    1
V2V0001    1
V2V0003    1
V2V0004    1
V2V0005    1
          ..
V2V0022    1
V2V0021    1
V2V0019    1
V2V0018    1
V2V0017    1
Name: count, Length: 798, dtype: int64

TEST:
business_id
V2V0002    1
V2V0015    1
V2V0020    1
V2V0026    1
V2V0033    1
          ..
V2V0973    1
V2V0974    1
V2V0978    1
V2V0980    1
V2V0982    1
Name: count, Length: 200, dtype: int64

city
TRAIN:
city
Ludhiana     110
Mumbai       105
Sehore       104
Jaipur       101
Surat        100
Hyderabad     96
Varanasi      94
Kochi         88
Name: count, dtype: int64

TEST:
city
Kochi        37
Varanasi     30
Hyderabad    29
Jaipur       24
Surat        24
Sehore       21
Mumbai       20
Ludhiana     15
Name: count, dtype: int64

country
TRAIN:
country
India    798
Name: count, dtype: int64

TEST:
country
India    200
Name: count, dtype: int64

sector
TRAIN:
sector
Retail            325
Other Services    265
Manufacturing     208
Name: count, dtype: int64

TEST:
s

In [36]:
for col in categorical_columns:

    train_values = set(
        train_df[col].dropna().unique()
    )

    test_values = set(
        test_df[col].dropna().unique()
    )

    unseen = test_values - train_values

    if unseen:
        print(
            f"{col}: unseen test values → {unseen}"
        )

business_id: unseen test values → {'V2V0727', 'V2V0787', 'V2V0123', 'V2V0619', 'V2V0836', 'V2V0178', 'V2V0026', 'V2V0459', 'V2V0696', 'V2V0369', 'V2V0907', 'V2V0173', 'V2V0079', 'V2V0364', 'V2V0072', 'V2V0854', 'V2V0717', 'V2V0914', 'V2V0877', 'V2V0095', 'V2V0785', 'V2V0372', 'V2V0878', 'V2V0431', 'V2V0247', 'V2V0518', 'V2V0396', 'V2V0020', 'V2V0569', 'V2V0263', 'V2V0630', 'V2V0356', 'V2V0741', 'V2V0920', 'V2V0198', 'V2V0792', 'V2V0230', 'V2V0889', 'V2V0109', 'V2V0066', 'V2V0148', 'V2V0503', 'V2V0782', 'V2V0827', 'V2V0953', 'V2V0941', 'V2V0201', 'V2V0955', 'V2V0082', 'V2V0406', 'V2V0357', 'V2V0779', 'V2V0684', 'V2V0767', 'V2V0315', 'V2V0534', 'V2V0341', 'V2V0080', 'V2V0746', 'V2V0278', 'V2V0328', 'V2V0296', 'V2V0374', 'V2V0980', 'V2V0918', 'V2V0890', 'V2V0942', 'V2V0284', 'V2V0336', 'V2V0820', 'V2V0143', 'V2V0257', 'V2V0323', 'V2V0735', 'V2V0103', 'V2V0188', 'V2V0423', 'V2V0679', 'V2V0121', 'V2V0703', 'V2V0956', 'V2V0852', 'V2V0924', 'V2V0151', 'V2V0099', 'V2V0884', 'V2V0978', 'V2V0107

Cleaning Operations & Final Validation

In [37]:
train_clean = train_df.copy()
test_clean = test_df.copy()

In [38]:
train_clean = train_clean.drop_duplicates()
test_clean = test_clean.drop_duplicates()

Save Cleaned Datasets

In [39]:
train_output = "/content/train_business_context_cleaned.csv"
test_output = "/content/test_business_context_cleaned.csv"

train_clean.to_csv(
    train_output,
    index=False
)

test_clean.to_csv(
    test_output,
    index=False
)

print("Saved:")
print(train_output)
print(test_output)

Saved:
/content/train_business_context_cleaned.csv
/content/test_business_context_cleaned.csv


In [41]:
from google.colab import files

files.download(train_output)
files.download(test_output)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Generate Data Quality Report

In [42]:
final_report = pd.DataFrame([
    {
        "dataset": "train",
        "rows": len(train_clean),
        "columns": len(train_clean.columns),
        "missing_cells": int(
            train_clean.isnull().sum().sum()
        ),
        "duplicate_rows": int(
            train_clean.duplicated().sum()
        ),
        "unique_business_ids": int(
            train_clean["business_id"].nunique()
        )
    },
    {
        "dataset": "test",
        "rows": len(test_clean),
        "columns": len(test_clean.columns),
        "missing_cells": int(
            test_clean.isnull().sum().sum()
        ),
        "duplicate_rows": int(
            test_clean.duplicated().sum()
        ),
        "unique_business_ids": int(
            test_clean["business_id"].nunique()
        )
    }
])

final_report

,dataset,rows,columns,missing_cells,duplicate_rows,unique_business_ids
0,train,798,31,0,0,798
1,test,200,31,0,0,200


In [43]:
final_report.to_csv(
    "/content/data_quality_report.csv",
    index=False
)

files.download(
    "/content/data_quality_report.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>